# Parte 1: Model Checking Proposicional

En esta parte implementarás las funciones de **model checking** para lógica proposicional.

## Objetivos
1. Generar todos los modelos posibles para un conjunto de átomos
2. Verificar satisfacibilidad de fórmulas
3. Verificar si una fórmula es tautología
4. Verificar si una base de conocimiento implica una consulta
5. Generar tablas de verdad

In [1]:
import sys
sys.path.insert(0, '..')

from src.logic_core import Atom, Not, And, Or, Implies, Iff, evaluate, get_atoms
from src.utils import formula_to_string, print_truth_table

## Paso 0: Familiarízate con el AST

Antes de implementar, experimenta con las clases de `logic_core.py`.

In [2]:
# Crear fórmulas
p = Atom('p')
q = Atom('q')

# p → q
f1 = Implies(p, q)
print(f"Fórmula: {formula_to_string(f1)}")
print(f"Átomos: {get_atoms(f1)}")
print(f"Evaluación con p=True, q=False: {evaluate(f1, {'p': True, 'q': False})}")
print(f"Evaluación con p=True, q=True: {evaluate(f1, {'p': True, 'q': True})}")

Fórmula: (p → q)
Átomos: frozenset({'p', 'q'})
Evaluación con p=True, q=False: False
Evaluación con p=True, q=True: True


In [3]:
# Fórmula más compleja: (p ∧ q) → (p ∨ r)
r = Atom('r')
f2 = Implies(And(p, q), Or(p, r))
print(f"Fórmula: {formula_to_string(f2)}")
print(f"Átomos: {get_atoms(f2)}")

Fórmula: ((p ∧ q) → (p ∨ r))
Átomos: frozenset({'p', 'r', 'q'})


### Bicondicional (↔)

El bicondicional `Iff(p, q)` es verdadero cuando ambos tienen el mismo valor de verdad: ambos verdaderos o ambos falsos. Se lee "p si y solo si q".

In [4]:
# Bicondicional: p ↔ q
f3 = Iff(p, q)
print(f"Fórmula: {formula_to_string(f3)}")
print(f"p=True,  q=True:  {evaluate(f3, {'p': True, 'q': True})}")
print(f"p=True,  q=False: {evaluate(f3, {'p': True, 'q': False})}")
print(f"p=False, q=True:  {evaluate(f3, {'p': False, 'q': True})}")
print(f"p=False, q=False: {evaluate(f3, {'p': False, 'q': False})}")

# Tabla de verdad del bicondicional
print("\nTabla de verdad de p ↔ q:")
print_truth_table(f3)

Fórmula: (p ↔ q)
p=True,  q=True:  True
p=True,  q=False: False
p=False, q=True:  False
p=False, q=False: True

Tabla de verdad de p ↔ q:
| p     | q     | (p ↔ q) |
|-------|-------|---------|
| False | False | True    |
| False | True  | False   |
| True  | False | False   |
| True  | True  | True    |


## Paso 1: Implementar `get_all_models`

Abre `src/model_checking.py` e implementa la función `get_all_models`.

**Hint:** Para n átomos, hay 2^n modelos posibles. Piensa en cómo los números del 0 al 2^n - 1 en binario te dan todas las combinaciones de True/False.

In [2]:
from src.model_checking import get_all_models

# Prueba tu implementación
models = get_all_models({'p', 'q','r','s'})
print(f"Número de modelos: {len(models)}")
for m in models:
    print(f"  {m}")

# Debe imprimir 4 modelos con todas las combinaciones

Número de modelos: 16
  {'r': False, 's': False, 'q': False, 'p': False}
  {'r': False, 's': False, 'q': False, 'p': True}
  {'r': False, 's': False, 'q': True, 'p': False}
  {'r': False, 's': False, 'q': True, 'p': True}
  {'r': False, 's': True, 'q': False, 'p': False}
  {'r': False, 's': True, 'q': False, 'p': True}
  {'r': False, 's': True, 'q': True, 'p': False}
  {'r': False, 's': True, 'q': True, 'p': True}
  {'r': True, 's': False, 'q': False, 'p': False}
  {'r': True, 's': False, 'q': False, 'p': True}
  {'r': True, 's': False, 'q': True, 'p': False}
  {'r': True, 's': False, 'q': True, 'p': True}
  {'r': True, 's': True, 'q': False, 'p': False}
  {'r': True, 's': True, 'q': False, 'p': True}
  {'r': True, 's': True, 'q': True, 'p': False}
  {'r': True, 's': True, 'q': True, 'p': True}


## Paso 2: Implementar `check_satisfiable`

Una fórmula es **satisfacible** si existe al menos un modelo donde es verdadera.

**Hint:** Usa `get_all_models` y `evaluate`.

In [3]:
from src.model_checking import check_satisfiable

# p ∧ q es satisfacible
sat, model = check_satisfiable(And(Atom('p'), Atom('q')))
print(f"p ∧ q es satisfacible: {sat}, modelo: {model}")

# p ∧ ¬p es insatisfacible
sat, model = check_satisfiable(And(Atom('p'), Not(Atom('p'))))
print(f"p ∧ ¬p es satisfacible: {sat}, modelo: {model}")

p ∧ q es satisfacible: True, modelo: {'q': True, 'p': True}
p ∧ ¬p es satisfacible: False, modelo: None


## Paso 3: Implementar `check_valid`

Una fórmula es **válida** (tautología) si es verdadera en todos los modelos.

**Hint:** φ es válida ⟺ ¬φ es insatisfacible.

In [4]:
from src.model_checking import check_valid

# p ∨ ¬p es tautología
print(f"p ∨ ¬p es válida: {check_valid(Or(Atom('p'), Not(Atom('p'))))}")

# p → p es tautología
print(f"p → p es válida: {check_valid(Implies(Atom('p'), Atom('p')))}")

# p no es tautología
print(f"p es válida: {check_valid(Atom('p'))}")

p ∨ ¬p es válida: True
p → p es válida: True
p es válida: False


## Paso 4: Implementar `check_entailment`

KB |= q si en todo modelo donde la KB es verdadera, q también lo es.

**Hint:** KB |= q ⟺ KB ∧ ¬q es insatisfacible.

In [5]:
from src.model_checking import check_entailment

# Modus ponens: {p → q, p} |= q
kb = [Implies(Atom('p'), Atom('q')), Atom('p')]
print(f"Modus ponens: {check_entailment(kb, Atom('q'))}")

# No se puede derivar q solo de p → q
kb = [Implies(Atom('p'), Atom('q'))]
print(f"Solo implicación: {check_entailment(kb, Atom('q'))}")

Modus ponens: True
Solo implicación: False


## Paso 5: Implementar `truth_table`

In [6]:
from src.model_checking import truth_table

# Tabla de verdad de p → q
table = truth_table(Implies(Atom('p'), Atom('q')))
for model, result in table:
    print(f"  {model} → {result}")

  {'q': False, 'p': False} → True
  {'q': False, 'p': True} → False
  {'q': True, 'p': False} → True
  {'q': True, 'p': True} → True


### Visualizar tabla de verdad con `print_truth_table`

La función `print_truth_table` imprime la tabla de verdad como una tabla ASCII formateada.

In [10]:
# Tabla de verdad de p → q (formateada)
print_truth_table(Implies(Atom('p'), Atom('q')))

print()

# Tabla con 3 átomos: (p ∧ q) → (p ∨ r)
print_truth_table(Implies(And(Atom('p'), Atom('q')), Or(Atom('p'), Atom('r'))))

| p     | q     | (p → q) |
|-------|-------|---------|
| False | False | True    |
| False | True  | True    |
| True  | False | False   |
| True  | True  | True    |

| p     | q     | r     | ((p ∧ q) → (p ∨ r)) |
|-------|-------|-------|---------------------|
| False | False | False | True                |
| False | False | True  | True                |
| False | True  | False | True                |
| False | True  | True  | True                |
| True  | False | False | True                |
| True  | False | True  | True                |
| True  | True  | False | True                |
| True  | True  | True  | True                |


## Aplicación: El Caso Criminal

Ahora aplica tu model checking al caso criminal.

In [7]:
# Base de conocimiento del caso criminal
kb_criminal = [
    Or(Atom('mayordomo_en_cocina'), Atom('mayordomo_en_biblioteca')),
    Implies(Atom('mayordomo_en_cocina'), Not(Atom('envenenó_copa'))),
    Implies(Not(Atom('llovía')), Not(Atom('jardín_mojado'))),
    Atom('jardín_mojado'),
    Or(Atom('envenenó_copa'), Atom('cocinera_envenenó')),
    Implies(Atom('cocinera_envenenó'), Atom('acceso_arsénico')),
    Not(Atom('acceso_arsénico')),
]

# Pregunta 1: ¿Llovía esa noche?
print(f"¿Llovía? {check_entailment(kb_criminal, Atom('llovía'))}")

# Pregunta 2: ¿La cocinera envenenó la copa?
print(f"¿Cocinera envenenó? {check_entailment(kb_criminal, Atom('cocinera_envenenó'))}")
print(f"¿Cocinera NO envenenó? {check_entailment(kb_criminal, Not(Atom('cocinera_envenenó')))}")

# Pregunta 3: ¿El mayordomo envenenó la copa?
print(f"¿Mayordomo envenenó? {check_entailment(kb_criminal, Atom('envenenó_copa'))}")

¿Llovía? True
¿Cocinera envenenó? False
¿Cocinera NO envenenó? True
¿Mayordomo envenenó? True


## Verificar con tests

Ejecuta los tests para validar tu implementación:

In [8]:
!cd .. && python -m pytest tests/test_model_checking.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- c:\Users\pifbl\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\pifbl\Documents\Uniandes\Inteligencia Artificial\Taller_3\Clue
configfile: pyproject.toml
plugins: anyio-4.7.0, typeguard-4.4.4
collecting ... collected 38 items

tests/test_model_checking.py::TestGetAllModels::test_single_atom PASSED  [  2%]
tests/test_model_checking.py::TestGetAllModels::test_two_atoms PASSED    [  5%]
tests/test_model_checking.py::TestGetAllModels::test_three_atoms PASSED  [  7%]
tests/test_model_checking.py::TestGetAllModels::test_empty_atoms PASSED  [ 10%]
tests/test_model_checking.py::TestGetAllModels::test_all_combinations_present PASSED [ 13%]
tests/test_model_checking.py::TestCheckSatisfiable::test_simple_satisfiable PASSED [ 15%]
tests/test_model_checking.py::TestCheckSatisfiable::test_contradiction_unsatisfiable PASSED [ 18%]
tests/test_mode